<a href="https://colab.research.google.com/github/paviayyala/AI-CyberSecurity-Research/blob/main/PII_MONITORING_DEMO_ENGINE.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
# ======================================================
# PII MONITORING DEMO ENGINE
# GDPR + DPDP Compliance Prototype
# ======================================================

# Install required libraries
!pip -q install presidio-analyzer pandas faker

from presidio_analyzer import AnalyzerEngine
import pandas as pd
import re
from datetime import datetime

print("\nInitializing PII Detection Engine...\n")

# Initialize analyzer
analyzer = AnalyzerEngine()

# ------------------------------------------------------
# INDIA SPECIFIC PII PATTERNS
# ------------------------------------------------------

aadhaar_pattern = r"\b[2-9]{1}[0-9]{3}\s?[0-9]{4}\s?[0-9]{4}\b"
pan_pattern = r"\b[A-Z]{5}[0-9]{4}[A-Z]{1}\b"
upi_pattern = r"\b[\w.-]+@[\w]+\b"

def detect_indian_pii(text):

    findings = []

    aadhaar = re.findall(aadhaar_pattern, text)
    pan = re.findall(pan_pattern, text)
    upi = re.findall(upi_pattern, text)

    for a in aadhaar:
        findings.append(("AADHAAR", a))

    for p in pan:
        findings.append(("PAN", p))

    for u in upi:
        findings.append(("UPI_ID", u))

    return findings


# ------------------------------------------------------
# PII SCANNING FUNCTION
# ------------------------------------------------------

def scan_for_pii(text):

    findings = []

    # Global PII detection
    results = analyzer.analyze(text=text, language="en")

    for r in results:
        findings.append({
            "entity": r.entity_type,
            "value": text[r.start:r.end],
            "confidence": round(r.score,3)
        })

    # Indian PII detection
    indian_results = detect_indian_pii(text)

    for entity,value in indian_results:
        findings.append({
            "entity": entity,
            "value": value,
            "confidence": 1.0
        })

    return findings


# ------------------------------------------------------
# TEST DATA (SIMULATED COMPANY DATA)
# ------------------------------------------------------

test_data = [

"Employee John Smith email john.smith@company.com phone +1 415 555 2671",

"Customer payment done using credit card 4111 1111 1111 1111",

"Employee Aadhaar number is 2345 6789 1234",

"Vendor PAN submitted as ABCDE1234F for invoice processing",

"Payment received through UPI rajesh@oksbi",

"Contact Rahul Sharma mobile +91 9876543210",

"Internal meeting tomorrow no sensitive data here"

]


print("===================================")
print("TEST DATA SCANNING STARTED")
print("===================================\n")

# ------------------------------------------------------
# SCAN DATA
# ------------------------------------------------------

log = []

for record in test_data:

    findings = scan_for_pii(record)

    if findings:

        print("PII DETECTED")
        print("Record:",record)

        for f in findings:

            print("  ->",f["entity"],":",f["value"])

            log.append({
                "timestamp":datetime.now(),
                "record":record,
                "pii_type":f["entity"],
                "pii_value":f["value"],
                "confidence":f["confidence"]
            })

        print("----------------------------------")

    else:
        print("No PII Found ->",record)
        print("----------------------------------")


# ------------------------------------------------------
# CREATE COMPLIANCE REPORT
# ------------------------------------------------------

df = pd.DataFrame(log)

print("\n===================================")
print("PII COMPLIANCE REPORT")
print("===================================\n")

display(df)


# Save report

df.to_csv("pii_compliance_report.csv",index=False)

print("\nReport generated -> pii_compliance_report.csv")

print("\nDemo Completed Successfully")

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 183.9/183.9 kB 3.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.0/2.0 MB 25.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.6/2.6 MB 34.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 105.9/105.9 kB 5.8 MB/s eta 0:00:00



Initializing PII Detection Engine...

✔ Download and installation successful
You can now load the package via spacy.load('en_core_web_lg')
⚠ Restart to reload dependencies
If you are in a Jupyter or Colab notebook, you may need to restart Python in
order to load all the package's dependencies. You can do this by selecting the
'Restart kernel' or 'Restart runtime' option.


TEST DATA SCANNING STARTED

PII DETECTED
Record: Employee John Smith email john.smith@company.com phone +1 415 555 2671
  -> EMAIL_ADDRESS : john.smith@company.com
  -> UK_NHS : 415 555 2671
  -> PERSON : John Smith
  -> PHONE_NUMBER : +1 415 555 2671
  -> URL : john.sm
  -> URL : company.com
  -> UPI_ID : john.smith@company
----------------------------------
PII DETECTED
Record: Customer payment done using credit card 4111 1111 1111 1111
  -> CREDIT_CARD : 4111 1111 1111 1111
  -> AADHAAR : 4111 1111 1111
----------------------------------
PII DETECTED
Record: Employee Aadhaar number is 2345 6789 1234
  -> AADHAAR : 2345 6789 1234
----------------------------------
PII DETECTED
Record: Vendor PAN submitted as ABCDE1234F for invoice processing
  -> PERSON : ABCDE1234F
  -> PAN : ABCDE1234F
----------------------------------
PII DETECTED
Record: Payment received through UPI rajesh@oksbi
  -> UPI_ID : rajesh@oksbi
----------------------------------
PII DETECTED
Record: Contact Rahul Shar

,timestamp,record,pii_type,pii_value,confidence
0,2026-03-08 13:30:40.124307,Employee John Smith email john.smith@company.c...,EMAIL_ADDRESS,john.smith@company.com,1.00
1,2026-03-08 13:30:40.124331,Employee John Smith email john.smith@company.c...,UK_NHS,415 555 2671,1.00
2,2026-03-08 13:30:40.124352,Employee John Smith email john.smith@company.c...,PERSON,John Smith,0.85
3,2026-03-08 13:30:40.124371,Employee John Smith email john.smith@company.c...,PHONE_NUMBER,+1 415 555 2671,0.75
4,2026-03-08 13:30:40.124390,Employee John Smith email john.smith@company.c...,URL,john.sm,0.50
5,2026-03-08 13:30:40.124408,Employee John Smith email john.smith@company.c...,URL,company.com,0.50
6,2026-03-08 13:30:40.124428,Employee John Smith email john.smith@company.c...,UPI_ID,john.smith@company,1.00
7,2026-03-08 13:30:40.151653,Customer payment done using credit card 4111 1...,CREDIT_CARD,4111 1111 1111 1111,1.00
8,2026-03-08 13:30:40.151666,Customer payment done using credit card 4111 1...,AADHAAR,4111 1111 1111,1.00
9,2026-03-08 13:30:40.163770,Employee Aadhaar number is 2345 6789 1234,AADHAAR,2345 6789 1234,1.00



Report generated -> pii_compliance_report.csv

Demo Completed Successfully


In [2]:
# ============================================================
# PII MONITORING ENGINE - COLAB PROTOTYPE
# Purpose:
# Detect Personally Identifiable Information (PII) in company data
# Supports GDPR / DPDP compliance proof-of-concept
# ============================================================


# ------------------------------------------------------------
# STEP 1 — INSTALL REQUIRED LIBRARIES
# ------------------------------------------------------------

# presidio-analyzer
# Microsoft open-source PII detection framework used by enterprises.
# It uses NLP + regex + ML to detect entities like email, phone, credit cards etc.

# pandas
# Used to store monitoring logs and generate compliance reports.

# faker
# Used to generate synthetic test data (safe testing).

!pip -q install presidio-analyzer presidio-anonymizer pandas faker


# ------------------------------------------------------------
# STEP 2 — IMPORT PYTHON LIBRARIES
# ------------------------------------------------------------

from presidio_analyzer import AnalyzerEngine
import pandas as pd
import re
from datetime import datetime


# ------------------------------------------------------------
# STEP 3 — INITIALIZE PII DETECTION ENGINE
# ------------------------------------------------------------

# AnalyzerEngine is the core Presidio component.
# It loads NLP models and entity recognizers.

analyzer = AnalyzerEngine()

print("PII Detection Engine Initialized\n")


# ------------------------------------------------------------
# STEP 4 — DEFINE INDIA-SPECIFIC PII REGEX PATTERNS
# ------------------------------------------------------------

# Presidio detects global PII but does not include some India identifiers.
# Therefore we define custom regex rules.

aadhaar_pattern = r"\b[2-9]{1}[0-9]{3}\s?[0-9]{4}\s?[0-9]{4}\b"

# Aadhaar format:
# 12 digits starting from 2-9

pan_pattern = r"\b[A-Z]{5}[0-9]{4}[A-Z]{1}\b"

# PAN format example:
# ABCDE1234F

upi_pattern = r"\b[\w.-]+@[\w]+\b"

# UPI format example:
# rahul@oksbi


# ------------------------------------------------------------
# FUNCTION 1 — DETECT INDIA-SPECIFIC PII
# ------------------------------------------------------------
def detect_indian_pii(text):

    """
    Purpose
    -------
    Detect India specific PII identifiers using regex rules.

    Why regex?
    ----------
    Aadhaar / PAN numbers follow fixed structural patterns.
    Regex provides fast pattern matching.

    Input
    -----
    text : string

    Output
    ------
    list of tuples
    (entity_type, detected_value)
    """

    findings = []

    # search Aadhaar
    aadhaar = re.findall(aadhaar_pattern, text)

    for a in aadhaar:
        findings.append(("AADHAAR", a))

    # search PAN
    pan = re.findall(pan_pattern, text)

    for p in pan:
        findings.append(("PAN", p))

    # search UPI
    upi = re.findall(upi_pattern, text)

    for u in upi:
        findings.append(("UPI_ID", u))

    return findings


# ------------------------------------------------------------
# FUNCTION 2 — SCAN TEXT FOR PII
# ------------------------------------------------------------
def scan_for_pii(text):

    """
    Core PII detection pipeline.

    Steps
    -----
    1. Use Presidio Analyzer
       Detect global PII entities.

    2. Use custom regex detectors
       Detect India-specific identifiers.

    3. Combine results into unified findings.

    Libraries Used
    --------------
    presidio-analyzer
        NLP + ML entity detection

    re
        Regex pattern matching

    Returns
    -------
    List of dictionaries describing PII findings
    """

    findings = []

    # ------------------------------
    # PRESIDIO DETECTION
    # ------------------------------
    results = analyzer.analyze(
        text=text,
        language="en"
    )

    for r in results:

        findings.append({
            "entity": r.entity_type,
            "value": text[r.start:r.end],
            "confidence": round(r.score,3)
        })


    # ------------------------------
    # INDIA SPECIFIC DETECTION
    # ------------------------------
    indian_results = detect_indian_pii(text)

    for entity,value in indian_results:

        findings.append({
            "entity": entity,
            "value": value,
            "confidence": 1.0
        })

    return findings


# ------------------------------------------------------------
# STEP 5 — TEST DATA (SIMULATED COMPANY DATA)
# ------------------------------------------------------------

# Example data representing:
# emails
# HR systems
# finance records
# customer transactions

company_records = [

"Employee John Smith email john.smith@company.com phone +1 415 555 2671",

"Customer payment made using credit card 4111 1111 1111 1111",

"Employee Aadhaar number is 2345 6789 1234",

"Vendor PAN number ABCDE1234F submitted",

"Payment done via UPI rajesh@oksbi",

"Customer Rahul Sharma mobile +91 9876543210"

]


print("Test Data Loaded")
print("Records:", len(company_records))
print("\nStarting PII scan...\n")


# ------------------------------------------------------------
# STEP 6 — MONITORING LOG STORAGE
# ------------------------------------------------------------

# This list will capture monitoring events.

log = []


# ------------------------------------------------------------
# STEP 7 — SCAN DATA
# ------------------------------------------------------------

for record in company_records:

    findings = scan_for_pii(record)

    if findings:

        print("PII DETECTED")
        print("Record:", record)

        for f in findings:

            print("   ->",f["entity"],":",f["value"])

            log.append({
                "timestamp": datetime.now(),
                "record": record,
                "pii_type": f["entity"],
                "pii_value": f["value"],
                "confidence": f["confidence"]
            })

        print("------------------------------------------------")


# ------------------------------------------------------------
# STEP 8 — CREATE COMPLIANCE REPORT
# ------------------------------------------------------------

df = pd.DataFrame(log)

print("\nCompliance Monitoring Table\n")

display(df)


# ------------------------------------------------------------
# STEP 9 — EXPORT REPORT
# ------------------------------------------------------------

df.to_csv("PII_Compliance_Report.csv",index=False)

print("\nReport saved: PII_Compliance_Report.csv")


# ------------------------------------------------------------
# DEMO SUMMARY
# ------------------------------------------------------------

print("\nPII Monitoring Demo Completed")

print("\nDetected Entities Include:")
print("- Email")
print("- Phone Numbers")
print("- Credit Cards")
print("- Aadhaar")
print("- PAN")
print("- UPI IDs")

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 4.5/4.5 MB 39.9 MB/s eta 0:00:00
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
pydrive2 1.21.3 requires cryptography<44, but you have cryptography 46.0.5 which is incompatible.
pyopenssl 24.2.1 requires cryptography<44,>=41.0.5, but you have cryptography 46.0.5 which is incompatible.


PII Detection Engine Initialized

Test Data Loaded
Records: 6

Starting PII scan...

PII DETECTED
Record: Employee John Smith email john.smith@company.com phone +1 415 555 2671
   -> EMAIL_ADDRESS : john.smith@company.com
   -> UK_NHS : 415 555 2671
   -> PERSON : John Smith
   -> PHONE_NUMBER : +1 415 555 2671
   -> URL : john.sm
   -> URL : company.com
   -> UPI_ID : john.smith@company
------------------------------------------------
PII DETECTED
Record: Customer payment made using credit card 4111 1111 1111 1111
   -> CREDIT_CARD : 4111 1111 1111 1111
   -> AADHAAR : 4111 1111 1111
------------------------------------------------
PII DETECTED
Record: Employee Aadhaar number is 2345 6789 1234
   -> AADHAAR : 2345 6789 1234
------------------------------------------------
PII DETECTED
Record: Vendor PAN number ABCDE1234F submitted
   -> PAN : ABCDE1234F
------------------------------------------------
PII DETECTED
Record: Payment done via UPI rajesh@oksbi
   -> UPI_ID : rajesh@oksbi
-

,timestamp,record,pii_type,pii_value,confidence
0,2026-03-08 13:33:38.448748,Employee John Smith email john.smith@company.c...,EMAIL_ADDRESS,john.smith@company.com,1.00
1,2026-03-08 13:33:38.448762,Employee John Smith email john.smith@company.c...,UK_NHS,415 555 2671,1.00
2,2026-03-08 13:33:38.448774,Employee John Smith email john.smith@company.c...,PERSON,John Smith,0.85
3,2026-03-08 13:33:38.448785,Employee John Smith email john.smith@company.c...,PHONE_NUMBER,+1 415 555 2671,0.75
4,2026-03-08 13:33:38.448795,Employee John Smith email john.smith@company.c...,URL,john.sm,0.50
5,2026-03-08 13:33:38.448821,Employee John Smith email john.smith@company.c...,URL,company.com,0.50
6,2026-03-08 13:33:38.448832,Employee John Smith email john.smith@company.c...,UPI_ID,john.smith@company,1.00
7,2026-03-08 13:33:38.459576,Customer payment made using credit card 4111 1...,CREDIT_CARD,4111 1111 1111 1111,1.00
8,2026-03-08 13:33:38.459590,Customer payment made using credit card 4111 1...,AADHAAR,4111 1111 1111,1.00
9,2026-03-08 13:33:38.468841,Employee Aadhaar number is 2345 6789 1234,AADHAAR,2345 6789 1234,1.00



Report saved: PII_Compliance_Report.csv

PII Monitoring Demo Completed

Detected Entities Include:
- Email
- Phone Numbers
- Credit Cards
- Aadhaar
- PAN
- UPI IDs
